In [12]:
import numpy as np
import qiskit
import openfermion
import pyscf
from openfermion import FermionOperator, QubitOperator


 H<sub>2</sub> molecule in minimal basis

In [13]:
# H2 molecule
from pyscf import gto, scf

# Build molecule
mol = gto.M(
    atom=[["H", (0.0, 0.0, 0.0)],
          ["H", (0.0, 0.0, 0.74)]],
    basis="sto-3g",
    unit="Angstrom",
)

# Hartree-Fock calculation
mf = scf.RHF(mol)
mf.kernel()

converged SCF energy = -1.11675930739643


np.float64(-1.1167593073964255)

In [14]:
import numpy as np
# Core Hamiltonian (kinetic + nuclear attraction)
h1 = mf.get_hcore()
print(h1)

[[-1.12095946 -0.95937577]
 [-0.95937577 -1.12095946]]


In [15]:
eri = mol.intor("int2e")
print(eri.shape)

(2, 2, 2, 2)


In [65]:
from pyscf import ao2mo

C = mf.mo_coeff

# One-electron integrals in MO basis
h1 = mf.get_hcore() # Core Hamiltonian (kinetic + nuclear attraction)
h1_mo = C.T @ h1 @ C

# Two-electron integrals in MO basis (Electron Repulsion Integrals)
eri_mo = ao2mo.kernel(mol, C)
eri_mo = ao2mo.restore(1, eri_mo, C.shape[1])
# h[p,q,r,s] = (ps|qr)
eri_mo = eri_mo.transpose(0, 2, 3, 1)

# 1 electron integrals
print(h1_mo)
# 2 electron integrals
print(eri_mo)

[[-1.25330979e+00  3.71049551e-17]
 [ 1.34841040e-17 -4.75068849e-01]]
[[[[ 6.74755927e-01 -3.75785642e-17]
   [-5.45047948e-17  1.81210462e-01]]

  [[-5.45047948e-17  1.81210462e-01]
   [ 6.63711401e-01  4.06319362e-16]]]


 [[[-3.75785642e-17  6.63711401e-01]
   [ 1.81210462e-01  5.57982113e-16]]

  [[ 1.81210462e-01  5.57982113e-16]
   [ 4.06319362e-16  6.97651504e-01]]]]


In [66]:
e_nuc = mf.energy_nuc()
e_nuc

np.float64(0.7151043390810812)

\begin{aligned}
a_i^\dagger =
\frac{1}{2}(X_i-iY_i)
\prod_{j<i}Z_j
\end{aligned}

\begin{aligned}
a_i =
\frac{1}{2}(X_i+iY_i)
\prod_{j<i}Z_j.
\end{aligned}


In [67]:
def creation(i):

    parity = 1
    for j in range(i):
        parity *= QubitOperator(f'Z{j}')

    return 0.5 * (QubitOperator(f'X{i}') - 1j * QubitOperator(f'Y{i}')) * parity

def annihilation(i):

    parity = 1
    for j in range(i):
        parity *= QubitOperator(f'Z{j}')

    return 0.5 * (QubitOperator(f'X{i}') + 1j * QubitOperator(f'Y{i}')) * parity


In [68]:
H_qubit = QubitOperator(()) * e_nuc

for i in range(2):
    for j in range(2):
        # α
        H_qubit += h1_mo[i, j] * creation(2*i) * annihilation(2*j)
        # β
        H_qubit += h1_mo[i, j] * creation(2*i+1) * annihilation(2*j+1)

# Two-electron part
for p in range(2):
    for q in range(2):
        for r in range(2):
            for s in range(2):

                value = 0.5 * eri_mo[p, q, r, s]
                # αα
                H_qubit += value * creation(2*p) * creation(2*q) * annihilation(2*r) * annihilation(2*s)

                # αβ
                H_qubit += value * creation(2*p) * creation(2*q+1) * annihilation(2*r+1) * annihilation(2*s)

                # βα
                H_qubit += value * creation(2*p+1) * creation(2*q) * annihilation(2*r) * annihilation(2*s+1)

                # ββ
                H_qubit += value * creation(2*p+1) * creation(2*q+1) * annihilation(2*r+1) * annihilation(2*s+1)

H_qubit

(-0.0970662681676322+0j) [] +
(-0.04530261550379919+0j) [X0 X1 Y2 Y3] +
(0.04530261550379919+0j) [X0 Y1 Y2 X3] +
(0.04530261550379919+0j) [Y0 X1 X2 Y3] +
(-0.04530261550379919+0j) [Y0 Y1 X2 X3] +
(0.17141282644776906+0j) [Z0] +
(0.1686889817036121+0j) [Z0 Z1] +
(0.12062523483390415+0j) [Z0 Z2] +
(0.16592785033770333+0j) [Z0 Z3] +
(0.17141282644776906+0j) [Z1] +
(0.16592785033770333+0j) [Z1 Z2] +
(0.12062523483390415+0j) [Z1 Z3] +
(-0.22343153690813364+0j) [Z2] +
(0.1744128761226153+0j) [Z2 Z3] +
(-0.22343153690813364+0j) [Z3]

In [73]:
from openfermion import get_sparse_operator

H_sparse = get_sparse_operator(H_qubit)
print(H_sparse)

<Compressed Sparse Column sparse matrix of dtype 'complex128'
	with 20 stored elements and shape (16, 16)>
  Coords	Values
  (0, 0)	(0.715104339081081+0j)
  (1, 1)	(0.2400354903089028+0j)
  (2, 2)	(0.24003549030890275+0j)
  (3, 3)	(0.4626181460271857+0j)
  (12, 3)	(0.18121046201519675+0j)
  (4, 4)	(-0.5382054475648963+0j)
  (5, 5)	(-0.5307733570014579+0j)
  (6, 6)	(-0.34956289498626125+0j)
  (9, 6)	(-0.18121046201519675+0j)
  (7, 7)	(0.35552070006763825+0j)
  (8, 8)	(-0.5382054475648963+0j)
  (6, 9)	(-0.18121046201519675+0j)
  (9, 9)	(-0.34956289498626125+0j)
  (10, 10)	(-0.5307733570014579+0j)
  (11, 11)	(0.35552070006763825+0j)
  (3, 12)	(0.18121046201519675+0j)
  (12, 12)	(-1.1167593073964253+0j)
  (13, 13)	(-0.4456158154821735+0j)
  (14, 14)	(-0.4456158154821735+0j)
  (15, 15)	(0.9231791809225394+0j)


In [74]:
np.linalg.eigvals(H_sparse.toarray().real)

array([-0.53077336, -0.16835243, -1.13728383,  0.48314267,  0.71510434,
        0.24003549,  0.24003549, -0.53820545, -0.53077336,  0.3555207 ,
       -0.53820545, -0.53077336,  0.3555207 , -0.44561582, -0.44561582,
        0.92317918])